# Flower Species and Accurate Color Detection

This notebook solves the problem of detecting 5 specific flowers (daisy, dandelion, rose, sunflower, tulip) using a highly accurate pre-trained model (`EfficientNetB0`).

Furthermore, it tackles the **Background Color Problem** by ensuring the model doesn't just guess the background color. We use **OpenCV GrabCut** to segment (cut out) the flower from the background, and then apply **K-Means Clustering** strictly on the flower's pixels to find its dominant, accurate color.

In [4]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from collections import Counter
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import warnings
warnings.filterwarnings('ignore')

## 1. Parameters Setup

In [5]:
# Define Image Size & Training Parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 5
DATA_DIR = 'flowers' # Ensure your dataset is in d:/CV_project/flowers/

# 5 specific flower classes
CLASS_NAMES = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

## 2. Data Loading & Augmentation
We load the images from the `flowers/` directory and use random augmentations to improve the model's robust visual understanding.

In [6]:
# Load datasets with 80-20 train-validation split
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Optional but recommended data augmentation to prevent overfitting
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal_and_vertical"),
  tf.keras.layers.RandomRotation(0.2),
  tf.keras.layers.RandomZoom(0.2),
])

# Prefetching for faster training
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.map(lambda x, y: (data_augmentation(x, training=True), y)).cache().prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Found 4317 files belonging to 5 classes.
Using 3454 files for training.
Found 4317 files belonging to 5 classes.
Using 863 files for validation.


## 3. Top-Tier Model Building (EfficientNetB0)
We use `EfficientNetB0` which is state-of-the-art for fast and highly accurate image classification. We freeze the base model and only train our custom Top Layers for our 5 flower classes.

In [7]:
# Load pre-trained EfficientNetB0 without top layers
base_model = EfficientNetB0(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')

# Freeze Base
base_model.trainable = False

# Attach Custom Top Layers
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=1e-3), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,184 (16.08 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## 4. Training Model

In [6]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('flower_species_base.h5', save_best_only=True)
]

# Initial Training Phase
print("Starting Initial Training...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks
)

Starting Initial Training...
Epoch 1/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 670ms/step - accuracy: 0.6820 - loss: 0.8211

108/108 ━━━━━━━━━━━━━━━━━━━━ 100s 870ms/step - accuracy: 0.7779 - loss: 0.6080 - val_accuracy: 0.9027 - val_loss: 0.2875
Epoch 2/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 867ms/step - accuracy: 0.8822 - loss: 0.3310

108/108 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.8746 - loss: 0.3416 - val_accuracy: 0.9131 - val_loss: 0.2602
Epoch 3/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 753ms/step - accuracy: 0.8987 - loss: 0.2734

108/108 ━━━━━━━━━━━━━━━━━━━━ 98s 910ms/step - accuracy: 0.8909 - loss: 0.2883 - val_accuracy: 0.9189 - val_loss: 0.2290
Epoch 4/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 607ms/step - accuracy: 0.9252 - loss: 0.2240

108/108 ━━━━━━━━━━━━━━━━━━━━ 84s 777ms/step - accuracy: 0.9181 - loss: 0.2298 - val_accuracy: 0.9258 - val_loss: 0.2276
Epoch 5/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 84s 774ms/step - accuracy: 0.9270 - loss: 0.2125 - val_accuracy: 0.9235 - val_loss: 0.2325
Epoch 6/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 575ms/step - accuracy: 0.9349 - loss: 0.1926

108/108 ━━━━━━━━━━━━━━━━━━━━ 78s 720ms/step - accuracy: 0.9372 - loss: 0.1920 - val_accuracy: 0.9293 - val_loss: 0.2188
Epoch 7/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 78s 727ms/step - accuracy: 0.9398 - loss: 0.1682 - val_accuracy: 0.9293 - val_loss: 0.2277
Epoch 8/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 580ms/step - accuracy: 0.9480 - loss: 0.1521

108/108 ━━━━━━━━━━━━━━━━━━━━ 78s 728ms/step - accuracy: 0.9450 - loss: 0.1571 - val_accuracy: 0.9328 - val_loss: 0.2171
Epoch 9/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 578ms/step - accuracy: 0.9520 - loss: 0.1470

108/108 ━━━━━━━━━━━━━━━━━━━━ 78s 728ms/step - accuracy: 0.9499 - loss: 0.1498 - val_accuracy: 0.9282 - val_loss: 0.2146
Epoch 10/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 80s 739ms/step - accuracy: 0.9519 - loss: 0.1355 - val_accuracy: 0.9316 - val_loss: 0.2167
Epoch 11/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 78s 726ms/step - accuracy: 0.9612 - loss: 0.1223 - val_accuracy: 0.9340 - val_loss: 0.2189
Epoch 12/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 79s 731ms/step - accuracy: 0.9676 - loss: 0.1000 - val_accuracy: 0.9316 - val_loss: 0.2238


## 5. Fine-Tuning for "Perfect" Accuracy
We unfreeze the top 20 layers of the EfficientNet base to let the model specialize meticulously on our 5 flowers with a tiny learning rate.

In [7]:
base_model.trainable = True

# Keep most layers frozen, unfreeze the top 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

callbacks_fine = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('flower_species_perfect.h5', save_best_only=True)
]

print("Starting Fine-Tuning Phase...")
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=callbacks_fine
)
model.save('flower_species_and_color_model.h5')


Starting Fine-Tuning Phase...
Epoch 1/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 699ms/step - accuracy: 0.8494 - loss: 0.4070

108/108 ━━━━━━━━━━━━━━━━━━━━ 102s 864ms/step - accuracy: 0.8506 - loss: 0.3991 - val_accuracy: 0.8980 - val_loss: 0.2691
Epoch 2/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 98s 904ms/step - accuracy: 0.8906 - loss: 0.3153 - val_accuracy: 0.8957 - val_loss: 0.2935
Epoch 3/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.9082 - loss: 0.2807 - val_accuracy: 0.8946 - val_loss: 0.2951
Epoch 4/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 100s 931ms/step - accuracy: 0.9117 - loss: 0.2544 - val_accuracy: 0.8957 - val_loss: 0.2896


## 6. Segregation (Background Cutout) & Color Extraction Algorithm

Here we solve your core problem: 
1. `segment_flower()` dynamically cuts out the flower using GrabCut.
2. `get_accurate_color()` measures the strict true RGB values inside the cutout using KMeans, avoiding background contamination.

In [8]:
def segment_flower(image_path):
    '''Cuts out the flower from background using OpenCV GrabCut.'''
    img = cv2.imread(image_path)
    if img is None: return None, None, None
    
    # For displaying, keep a copy in RGB
    orig_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    mask = np.zeros(img.shape[:2], np.uint8)
    bgdModel = np.zeros((1, 65), np.float64)
    fgModel = np.zeros((1, 65), np.float64)
    
    # Bounding box covering standard central region where flowers usually sit
    h, w = img.shape[:2]
    rect = (int(w*0.1), int(h*0.1), int(w*0.8), int(h*0.8))
    
    cv2.grabCut(img, mask, rect, bgdModel, fgModel, 5, cv2.GC_INIT_WITH_RECT)
    
    # Where mask is 2 or 0 (background), set to 0, otherwise 1 (foreground)
    mask2 = np.where((mask==2)|(mask==0), 0, 1).astype('uint8')
    
    # Calculate cutout
    cutout = orig_rgb * mask2[:, :, np.newaxis]
    return orig_rgb, cutout, mask2

def get_accurate_color(image_rgb, mask):
    '''Calculates precise RGB of only the flower (foreground) via K-Means.'''
    # Extract foreground pixels safely
    pixels = image_rgb.reshape(-1, 3)
    mask_flat = mask.reshape(-1)
    fg_pixels = pixels[mask_flat == 1]
    
    if len(fg_pixels) == 0:
        return (0, 0, 0)
        
    # Use K-Means clustering to find 3 dominant colors inside the flower itself
    kmeans = KMeans(n_clusters=3, random_state=42)
    kmeans.fit(fg_pixels)
    
    # Get largest cluster (the single most dominant color)
    counts = Counter(kmeans.labels_)
    dominant_cluster = counts.most_common(1)[0][0]
    dominantColor = kmeans.cluster_centers_[dominant_cluster]
    
    return tuple(map(int, dominantColor))

## 7. The End-to-End Master Function
Combines Image Classification prediction with Advanced Color Parsing!

In [ ]:
def predict_and_analyze(image_path):
    # 1. Image Classification via EfficientNet
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_arr = tf.keras.preprocessing.image.img_to_array(img)
    img_arr = tf.expand_dims(img_arr, 0) # Create batch
    
    preds = model.predict(img_arr)
    confidence = 100 * np.max(preds)
    species = CLASS_NAMES[np.argmax(preds)]
    
    # 2. Extract Color via Segmentation Map
    orig, cutout, mask = segment_flower(image_path)
    dom_rgb = get_accurate_color(orig, mask)